In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time

import tensorflow as tf

import sheets.basicmodel as basicmodel
import sheets.utils.plotting as plotting
from sheets.constants import *
from sheets.dataloader import MAESTRODataLoader
from sheets.dataset_builder import build_tf_dataset
from sheets.helpers import *
from sheets.preprocessors import CQTPreprocessor, MELPreprocessor
from sheets.utils.pianoroll_audio import midi_object_to_playable, numpy_to_midi_object
from sheets.utils.simple_prediction import *

# Build Training & Validation Datasets

In [ ]:
dataloader = MAESTRODataLoader(year='all')

preprocessor = CQTPreprocessor()

train_ds = build_tf_dataset(
    dataloader,
    preprocessor,
    split='train',
    batch_size=8
)

val_ds = build_tf_dataset(
    dataloader,
    preprocessor,
    split='validation',
    batch_size=8
)

# Build & Compile Basic Model

In [ ]:
basic_mdl = basicmodel.initialize_model()
basic_mdl = basicmodel.compile_model(basic_mdl)

# Configure Early Stopping

In [ ]:
es = tf.keras.callbacks.EarlyStopping(
    patience=5,
    monitor='val_loss'
)

# Configure Model Checkpointing

In [ ]:
timestamp = time.strftime("%Y%m%d-%H%M%S")
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f'./models/{timestamp}.weights.h5',
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True
)

# Train Basic Model

In [ ]:
basic_history = basic_mdl.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[es, checkpoint_callback],
)

# Save Trained Model

In [ ]:
timestamp = time.strftime("%Y%m%d-%H%M%S")

model_path = os.path.join(
    './models',
    f'{timestamp}.h5'
)

basic_mdl.save(model_path)

# Plot Basic History

In [ ]:
plotting.plot_history(basic_history, metric_name='fbeta')

# Run Basic Model Predictions

In [ ]:
predict_simple(basic_mdl)
predict_simple_triples(basic_mdl)
predict_complex(basic_mdl)